# SpaceForge RT-GWN v12 -- Colab runner and report

Runs `train_v12.py` (Relational Telescope Graph WaveNet with competing-risk
hazards) on the Drive-hosted dataset produced by `feature_engineering_v2.py`,
then renders the result graphs.

Layout mirrors the v11 runner:
- the training script lives at `MyDrive/SpaceForgeData/scripts/train_v12.py`
  (or upload it to `/content/` and set `SCRIPT_PATH`)
- data root and output root are configured in the cell below
- every output CSV/JSON is loaded from the synced Drive output folder

Order of cells: config -> run -> metrics tables -> training curves ->
timing plots -> calibration -> survival curves -> learned adjacency ->
XAI (typed edges, features, acute-vs-chronic).


In [ ]:
# --- 1. Configuration -------------------------------------------------------
from pathlib import Path

MOUNT          = "/content/gdrive"
DRIVE_BASE     = f"{MOUNT}/MyDrive/SpaceForgeData"

# Where train_v12.py lives (Drive copy preferred, /content fallback).
SCRIPT_PATH    = f"{DRIVE_BASE}/scripts/train_v12.py"

# Data root: the feature_engineering_v2 export of the NEW 100-config dataset.
DRIVE_ROOT     = f"{DRIVE_BASE}/spaceforge-cleaned2/sf-cleaned-2"

# Where train_v12.py syncs its outputs.
OUTPUT_ROOT    = f"{DRIVE_BASE}/spaceforge-cleaned2/rtgwn_xai_outputs_v12"

# QUICK_RUN caps windows and uses one seed -- use it to smoke-test the whole
# pipeline (about 20-30 min) before a full multi-seed run.
QUICK_RUN      = True

CFG_OVERRIDES = {
    "drive_root":        DRIVE_ROOT,
    "drive_output_root": OUTPUT_ROOT,
}
if QUICK_RUN:
    CFG_OVERRIDES.update({
        "seeds": [42],
        "model_keys": ["B"],
        "max_windows_per_split": 4000,
        "max_epochs": 15,
        "xai_samples_per_cause": 3,
    })
print(CFG_OVERRIDES)


In [ ]:
# --- 2. Mount Drive and stage the script ------------------------------------
import os, shutil, json

from google.colab import drive
drive.mount(MOUNT)

local_script = "/content/train_v12.py"
if os.path.exists(SCRIPT_PATH):
    shutil.copy2(SCRIPT_PATH, local_script)
    print("staged", SCRIPT_PATH)
elif os.path.exists(local_script):
    print("using existing", local_script)
else:
    raise FileNotFoundError(
        f"train_v12.py not found at {SCRIPT_PATH} or {local_script}. "
        "Upload it or fix SCRIPT_PATH.")

overrides_path = "/content/v12_cfg_overrides.json"
with open(overrides_path, "w") as f:
    json.dump(CFG_OVERRIDES, f, indent=2)
os.environ["SF_V12_CFG_JSON"] = overrides_path
print("overrides written to", overrides_path)


In [ ]:
# --- 3. Run training (streams the console log live) --------------------------
import subprocess, sys

proc = subprocess.Popen(
    [sys.executable, "-u", local_script],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, env=dict(os.environ),
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print("exit code:", proc.returncode)
assert proc.returncode == 0, "training failed -- scroll up for the first error"


In [ ]:
# --- 4. Load outputs ---------------------------------------------------------
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
from pathlib import Path

OUT = Path(OUTPUT_ROOT)

def load_csv(name):
    p = OUT / name
    if p.exists():
        return pd.read_csv(p)
    print(f"[missing] {name}")
    return None

summary = json.loads((OUT / "run_summary.json").read_text()) if (OUT / "run_summary.json").exists() else None
comparison = load_csv("model_comparison.csv")
print("run_tag:", summary["run_tag"] if summary else "n/a")
print("configs:", summary["n_configs"] if summary else "n/a",
      " runs:", summary["n_runs"] if summary else "n/a",
      " windows:", summary["n_windows"] if summary else "n/a")
comparison


In [ ]:
# --- 5. Training curves ------------------------------------------------------
import glob

hist_files = sorted(glob.glob(str(OUT / "training_history_*.csv")))
if hist_files:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for hf in hist_files:
        h = pd.read_csv(hf)
        tag = Path(hf).stem.replace("training_history_", "")
        axes[0].plot(h["epoch"], h["train_loss"], label=tag)
        axes[1].plot(h["epoch"], h["val_score"], label=tag)
        axes[2].plot(h["epoch"], h["val_auroc"], label=tag)
    for ax, t in zip(axes, ["train loss", "val composite score", "val AUROC"]):
        ax.set_title(t); ax.set_xlabel("epoch"); ax.legend(fontsize=7)
    plt.tight_layout(); plt.show()


In [ ]:
# --- 6. Timing: per-horizon F1 and TTF-bin quality ---------------------------
hor = load_csv("per_horizon_metrics.csv")
if hor is not None:
    fig, ax = plt.subplots(figsize=(10, 4))
    for (m, s), g in hor.groupby(["model", "seed"]):
        g = g.sort_values("horizon")
        ax.plot(g["horizon"], g["f1"], marker="o", label=f"{m}_s{s}")
    ax.set_xlabel("horizon (ticks)"); ax.set_ylabel("F1")
    ax.set_title("early-warning F1 vs horizon (from hazard CDF, monotone by construction)")
    ax.set_xscale("log"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

bins_df = load_csv("ttf_bin_metrics.csv")
reg_df  = load_csv("ttf_regression_metrics.csv")
if bins_df is not None: display(bins_df)
if reg_df  is not None: display(reg_df)


In [ ]:
# --- 7. Calibration (reliability diagram) ------------------------------------
cal = load_csv("hazard_calibration.csv")
if cal is not None:
    fig, ax = plt.subplots(figsize=(5, 5))
    for (m, s), g in cal.groupby(["model", "seed"]):
        ax.plot(g["mean_prob"], g["frac_pos"], marker="o", label=f"{m}_s{s}")
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
    ax.set_xlabel("predicted P(fail <= 240)"); ax.set_ylabel("observed frequency")
    ax.set_title("binary risk calibration"); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()


In [ ]:
# --- 8. Survival curves (sample of test windows) -----------------------------
surv = load_csv("survival_curves_sample.csv")
if surv is not None:
    fig, ax = plt.subplots(figsize=(9, 5))
    for wid, g in surv.groupby("window_id"):
        g = g.sort_values("horizon")
        failed = g["y_fail"].iloc[0] == 1
        color = "tab:red" if failed else "tab:green"
        ax.plot(g["horizon"], g["survival"], color=color, alpha=0.35, lw=1)
        if failed and g["ttf_ticks"].iloc[0] >= 0:
            ax.axvline(g["ttf_ticks"].iloc[0], color="tab:red", alpha=0.08)
    ax.set_xlabel("ticks ahead"); ax.set_ylabel("predicted survival S(t)")
    ax.set_title("sample survival curves (red = failing runs, green = alive)")
    ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()


In [ ]:
# --- 9. Learned adaptive adjacency per relation ------------------------------
import glob as _g

adj_files = sorted(_g.glob(str(OUT / "adaptive_adjacency_*.csv")))
if adj_files:
    show = adj_files[:3]
    fig, axes = plt.subplots(1, len(show), figsize=(6 * len(show), 5))
    if len(show) == 1:
        axes = [axes]
    for ax, af in zip(axes, show):
        adj = pd.read_csv(af, index_col=0)
        im = ax.imshow(adj.values, cmap="viridis", aspect="auto")
        ax.set_xticks(range(len(adj.columns))); ax.set_xticklabels(adj.columns, rotation=90, fontsize=7)
        ax.set_yticks(range(len(adj.index)));  ax.set_yticklabels(adj.index, fontsize=7)
        ax.set_title(Path(af).stem.replace("adaptive_adjacency_", ""), fontsize=9)
        fig.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout(); plt.show()


In [ ]:
# --- 10. XAI: typed edges, features, acute-vs-chronic ------------------------
edges = load_csv("xai_relation_edge_masks.csv")
if edges is not None and len(edges):
    top = (edges.groupby(["cause", "relation", "src", "dst"])["weight"]
                 .mean().reset_index()
                 .sort_values("weight", ascending=False))
    for cause, g in top.groupby("cause"):
        g = g.head(8)
        fig, ax = plt.subplots(figsize=(8, 2.5))
        labels = [f"{r.relation}: {r.src} -> {r.dst}" for r in g.itertuples()]
        ax.barh(labels[::-1], g["weight"].values[::-1])
        ax.set_title(f"top typed edges: {cause}"); ax.set_xlabel("mean mask weight")
        plt.tight_layout(); plt.show()

feats = load_csv("xai_feature_masks.csv")
if feats is not None and len(feats):
    topf = (feats.groupby(["cause", "node", "feature"])["weight"]
                  .mean().reset_index().sort_values("weight", ascending=False))
    display(topf.groupby("cause").head(5).reset_index(drop=True))

ac = load_csv("xai_acute_chronic.csv")
if ac is not None and len(ac):
    fig, ax = plt.subplots(figsize=(7, 4))
    for cause, g in ac.groupby("cause"):
        ax.scatter(g["fast_weight"], g["slow_weight"], label=cause, alpha=0.7)
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("fast-stream weight (acute)"); ax.set_ylabel("slow-stream weight (chronic)")
    ax.set_title("acute vs chronic attribution per explained failure")
    ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()
    display(ac.groupby("cause")["chronic_ratio"].describe().round(3))

faith = load_csv("xai_faithfulness_v12.csv")
if faith is not None and len(faith):
    display(faith)


In [ ]:
# --- 11. Optional: compare with a v11 run ------------------------------------
# Point V11_SUMMARY at a v11 run_summary.json to get a side-by-side table.
V11_SUMMARY = f"{DRIVE_BASE}/spaceforge-cleaned2/graphwavenet_xai_outputs_v11/run_summary.json"

import os
if os.path.exists(V11_SUMMARY) and summary is not None:
    v11 = json.loads(Path(V11_SUMMARY).read_text())
    rows = []
    def first_result(d):
        res = d.get("results", {})
        return next(iter(res.values())) if res else {}
    r11, r12 = first_result(v11), first_result(summary)
    for k in ["auroc", "ap", "f1", "horizon_macro_f1", "ttf_bin_macro_f1",
              "cause_macro_f1_near", "ttf_mae_ticks", "ttf_mae_ticks_near", "ece"]:
        rows.append({"metric": k, "v11": r11.get(k), "v12": r12.get(k)})
    display(pd.DataFrame(rows))
else:
    print("v11 summary not found (or v12 summary missing) -- skipping comparison")


## Reading the results

- **Horizon F1 curve**: v12 horizons come from one survival curve, so they are
  monotone by construction; compare the short-horizon (5-20 tick) end against
  v11's `per_horizon_metrics.csv` -- that is where the hazard head should win.
- **Calibration**: v11 had no calibrated risk; the reliability diagram plus the
  `ece` column is a new capability. If the curve sags below the diagonal the
  model over-predicts risk.
- **Acute vs chronic**: points near the x-axis are acute failures (fast stream
  carried the explanation), near the y-axis chronic ones (slow stream, i.e.
  depletion/drift). Expect battery_power to lean acute and effusion causes
  driven by SourceInventory depletion to lean chronic.
- **Typed edges**: the faithfulness table reports how many of the top explained
  edges fall on the expected physical path for each cause.
